# Tournament Data Preparation

This notebook prepares the tournament-level data used in the project.

It downloads ATP and WTA match data for 2020-2026, resolves player-identity inconsistencies, filters tournaments that do not follow the standard knockout structure used in the analysis, validates the cleaned match data, and aggregates match-level records into one row per player-tournament.

The final output is `player_tournament_data.csv`, which is used in the subsequent data-integration and modeling pipeline.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import os

# ============================================================
# Project paths
# ============================================================

PROJECT_DIR = Path.cwd()

# Support running from the project root or data_code/
if PROJECT_DIR.name == "data_code":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"

RAW_MATCHES_DIR = DATA_DIR / "raw_matches"
CLEAN_MATCHES_DIR = DATA_DIR / "clean_matches"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

RAW_MATCHES_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_MATCHES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Raw files
# ============================================================

ATP_RAW_MATCHES_FILE = RAW_MATCHES_DIR / "atp_matches_all.csv"
WTA_RAW_MATCHES_FILE = RAW_MATCHES_DIR / "wta_matches_all.csv"

ATP_RAW_PLAYERS_FILE = RAW_MATCHES_DIR / "atp_active_players.csv"
WTA_RAW_PLAYERS_FILE = RAW_MATCHES_DIR / "wta_active_players.csv"

RAW_MATCH_FILES = {
    "atp": ATP_RAW_MATCHES_FILE,
    "wta": WTA_RAW_MATCHES_FILE,
}

RAW_PLAYER_FILES = {
    "atp": ATP_RAW_PLAYERS_FILE,
    "wta": WTA_RAW_PLAYERS_FILE,
}
# ============================================================
# Clean files
# ============================================================

ATP_CLEAN_MATCHES_FILE = CLEAN_MATCHES_DIR / "atp_matches_all_clean.csv"
WTA_CLEAN_MATCHES_FILE = CLEAN_MATCHES_DIR / "wta_matches_all_clean.csv"

ATP_CLEAN_PLAYERS_FILE = CLEAN_MATCHES_DIR / "atp_active_players_clean.csv"
WTA_CLEAN_PLAYERS_FILE = CLEAN_MATCHES_DIR / "wta_active_players_clean.csv"

CLEAN_MATCH_FILES = {
    "atp": ATP_CLEAN_MATCHES_FILE,
    "wta": WTA_CLEAN_MATCHES_FILE,
}

CLEAN_PLAYER_FILES = {
    "atp": ATP_CLEAN_PLAYERS_FILE,
    "wta": WTA_CLEAN_PLAYERS_FILE,
}

# ============================================================
# Final tournament-level dataset
# ============================================================

PLAYER_TOURNAMENT_FILE = PROCESSED_DATA_DIR / "player_tournament_data.csv"

# ============================================================
# General settings
# ============================================================

START_YEAR = 2020
END_YEAR = 2026

YEARS = range(START_YEAR, END_YEAR + 1)
TOURS = ["atp", "wta"]

## Raw Match Data Collection

Historical ATP and WTA player and match data for 2020-2026 were obtained from the Tennis Sackmann Archive hosted on Hugging Face.

For each tour, the yearly match files were combined into a single raw match dataset. The player table was restricted to players appearing in at least one match during the study period.

In [ ]:
# Download data without filtering

for tour in TOURS:

    print(f"\n========== {tour.upper()} ==========")

    # ---------- Players ----------
    players_url = f"https://huggingface.co/datasets/Aneeshers/tennis-sackmann-archive/resolve/main/{tour}/{tour}_players.csv"

    players_df = pd.read_csv(players_url)

    players_df["full_name"] = (
            players_df["name_first"].fillna("").astype(str).str.strip()
            + " "
            + players_df["name_last"].fillna("").astype(str).str.strip()
    ).str.strip()

    # ---------- Matches ----------
    all_matches = []

    for year in YEARS:
        url = f"https://huggingface.co/datasets/Aneeshers/tennis-sackmann-archive/resolve/main/{tour}/{tour}_matches_{year}.csv"

        try:
            df = pd.read_csv(url)
            df["year"] = year
            all_matches.append(df)
            print(f"Downloaded {tour}_matches_{year}.csv")

        except Exception as e:
            print(f"Could not download {tour} {year}: {e}")

    # Combine all matches
    all_matches_df = pd.concat(all_matches, ignore_index=True)

    all_matches_df.to_csv(
        RAW_MATCH_FILES[tour],
        index=False
    )

    print(f"Saved {RAW_MATCH_FILES[tour]}")
    print(f"Matches: {len(all_matches_df)}")

    # ---------- Filter active players ----------
    active_player_ids = pd.concat([
        all_matches_df["winner_id"],
        all_matches_df["loser_id"]
    ]).dropna().unique()

    active_players_df = players_df[
        players_df["player_id"].isin(active_player_ids)
    ].copy()

    active_players_df["tour"] = tour.upper()

    # Save only active players
    active_players_df.to_csv(
        RAW_PLAYER_FILES[tour],
        index=False
    )

    print(f"Saved {RAW_PLAYER_FILES[tour]}")
    print(f"Active players: {len(active_players_df)}")

## Player Identity Validation

Before cleaning the match data, player identifiers were checked for consistency across the ATP and WTA sources.

We examined cases in which one player ID was associated with multiple names and cases in which the same player name appeared under multiple IDs. The identified duplicate identities were manually reviewed and used to define the canonical ID mappings applied during cleaning.

In [ ]:
atp_active_players_df = pd.read_csv(ATP_RAW_PLAYERS_FILE)
wta_active_players_df = pd.read_csv(WTA_RAW_PLAYERS_FILE)
active_players_df = pd.concat(
    [atp_active_players_df, wta_active_players_df],
    ignore_index=True
)


atp_matches_df = pd.read_csv(ATP_RAW_MATCHES_FILE)
wta_matches_df = pd.read_csv(WTA_RAW_MATCHES_FILE)
atp_matches_df["tour"] = "ATP"
wta_matches_df["tour"] = "WTA"
all_matches_df = pd.concat(
    [atp_matches_df, wta_matches_df],
    ignore_index=True
)

In [ ]:
def check_duplicated_in_data(active_players_df, id_col_name="player_id"):
    print(f"duplicated? {active_players_df.duplicated().any()}")

    # 1. Same player_id with multiple full names
    player_ids_with_multiple_names = (
        active_players_df
        .groupby(id_col_name)["full_name"]
        .nunique()
    )

    player_ids_with_multiple_names = (
        player_ids_with_multiple_names[
            player_ids_with_multiple_names > 1
            ]
    )

    print("Player IDs with multiple full names:")
    print(player_ids_with_multiple_names)

    if not player_ids_with_multiple_names.empty:
        print(
            active_players_df[
                active_players_df[id_col_name].isin(
                    player_ids_with_multiple_names.index
                )
            ][[id_col_name, "full_name", "tour"]]
            .drop_duplicates()
            .sort_values([id_col_name, "full_name"])
        )

    # 2. Same full_name with multiple player_ids

    names_with_multiple_ids = (
        active_players_df
        .groupby(["tour", "full_name"])[id_col_name]
        .nunique()
    )

    names_with_multiple_ids = names_with_multiple_ids[
        names_with_multiple_ids > 1
        ]

    print("\nFull names with multiple player IDs:")
    print(names_with_multiple_ids)

    if not names_with_multiple_ids.empty:
        problematic_names = (
            names_with_multiple_ids
            .reset_index()[["tour", "full_name"]]
        )

        details = active_players_df.merge(
            problematic_names,
            on=["tour", "full_name"],
            how="inner"
        )

        print(
            details[
                ["tour", "full_name", id_col_name]
            ]
            .drop_duplicates()
            .sort_values(["tour", "full_name", id_col_name])
        )

        display(details.sort_values(["full_name", "tour", id_col_name]))
        print("")

        # 3. Exactly one row per player (tour + player_id)
        print(
            "Exactly one row per player:",
            not active_players_df.duplicated(
                subset=["tour", id_col_name]
            ).any()
        )

        duplicate_players = active_players_df[
            active_players_df.duplicated(
                subset=["tour", id_col_name],
                keep=False
            )
        ]

        if not duplicate_players.empty:
            print("\nDuplicate rows:")
            print(
                duplicate_players[
                    ["tour", id_col_name, "full_name"]
                ]
                .drop_duplicates()
                .sort_values(["tour", id_col_name, "full_name"])
            )

In [ ]:
check_duplicated_in_data(active_players_df)

## Tournament Format Inspection

Before filtering the match data, we examined the round structures observed in the raw tournaments.

The source data contains standard knockout tournaments as well as team competitions, round-robin events, and tournaments with special rounds such as bronze-medal matches. Since tournament progression is used as the main performance measure in this project, only comparable standard knockout formats are retained.

In [ ]:
tournament_rounds = (
    all_matches_df
    .groupby(["tour", "tourney_name", "year"])
    .agg(
        round_pattern=(
            "round",
            lambda values: tuple(
                sorted(values.dropna().unique())
            )
        ),
        tourney_level=("tourney_level", "first")
    )
    .reset_index()
)

tournament_rounds["tournament_year"] = (
    tournament_rounds["tourney_name"]
    + " ("+ tournament_rounds["year"].astype(str)+ ")"
)

round_patterns = (
    tournament_rounds
    .groupby(["tour", "round_pattern"])
    .agg(
        num_tournament_years=(
            "tournament_year",
            "count"
        ),
        tournament_years=(
            "tournament_year",
            lambda values: sorted(values.unique())
        )
    )
    .reset_index()
    .sort_values(
        ["tour", "num_tournament_years"],
        ascending=[True, False]
    )
)

display(
    round_patterns.sort_values(
        ["num_tournament_years", "tour"],
        ascending=[True, True]
    )
)

## Match Data Cleaning

The raw match data is cleaned before tournament-level aggregation.

To support comparable tournament-progression analysis, we retain only standard knockout tournaments. Team competitions, round-robin tournaments, and events containing bronze-medal rounds are excluded.

Player IDs are mapped to canonical identifiers, entry codes are standardized (`Alt` to `ALT`, `W` to `WC`, and `L` to `LL`), surface capitalization is normalized, and missing playing-hand values are represented as `U` for unknown.

Four duplicated player identities were found under different IDs. Their old IDs are mapped to canonical IDs in both the player and match datasets, and duplicate player records are resolved by retaining the most complete record.

Globally unique `player_key` and `tournament_key` identifiers are created to prevent ATP/WTA ID collisions.

One completed match had `minutes = 0`; because the correct duration was unavailable, it is treated as missing.

In [ ]:
TEAM_TOURNAMENTS = [
    "Atp Cup",
    "United Cup",
    "Laver Cup",
]

EXCLUDED_TOURNAMENTS = [
    "Grampians Trophy",
]

SPECIAL_ROUNDS = ["RR", "BR"]

ENTRY_FIXES = {
    "W": "WC",
    "L": "LL",
    "Alt": "ALT",
}

# Old ID -> canonical ID
PLAYER_ID_FIXES = {
    "atp": {
        209870: 211326,  # Gunawan Trismuwantara
        211776: 212021,  # Martin Landaluce
    },
    "wta": {
        260672: 241717,  # Johanne Christine Svendsen
        266531: 239456,  # Tiantsoa Sarah Rakotomanga Rajaonah
    },
}

EXPECTED_ROUNDS = {
    "R128",
    "R64",
    "R32",
    "R16",
    "QF",
    "SF",
    "F",
}

EXPECTED_SURFACES = {
    "Hard",
    "Clay",
    "Grass",
}


In [ ]:
# Download and clean the ATP and WTA data

for tour in TOURS:

    print(f"\n========== {tour.upper()} ==========")

    # ---------- Players ----------
    players_url = f"https://huggingface.co/datasets/Aneeshers/tennis-sackmann-archive/resolve/main/{tour}/{tour}_players.csv"

    players_df = pd.read_csv(players_url)

    players_df["player_id"] = pd.to_numeric(players_df["player_id"], errors="coerce").astype("Int64")

    # Fix duplicate IDs that refer to the same player
    players_df["player_id"] = players_df["player_id"].replace(PLAYER_ID_FIXES[tour])

    players_df["full_name"] = (
        players_df["name_first"].fillna("").astype(str).str.strip()
        + " "
        + players_df["name_last"].fillna("").astype(str).str.strip()
    ).str.strip()

    # Missing hand means unknown, so standardize it as "U"
    if "hand" in players_df.columns:
        players_df["hand"] = (
            players_df["hand"]
            .astype("string")
            .str.strip()
            .str.upper()
            .fillna("U")
            .replace("", "U")
        )

    players_df["tour"] = tour.upper()

    players_df["player_key"] = (
        players_df["tour"]
        + "_"
        + players_df["player_id"].astype("string")
    )

    # If two player rows became one after the ID correction,
    # keep the row with more available information
    player_info_columns = [
        "name_first",
        "name_last",
        "hand",
        "dob",
        "ioc",
        "height",
        "wikidata_id",
    ]

    existing_info_columns = [
        col for col in player_info_columns
        if col in players_df.columns
    ]

    players_df["_known_fields"] = (
        players_df[existing_info_columns]
        .notna()
        .sum(axis=1)
    )

    players_df = (
        players_df
        .sort_values("_known_fields", ascending=False)
        .drop_duplicates(subset=["player_key"], keep="first")
        .drop(columns="_known_fields")
        .reset_index(drop=True)
    )

    # ---------- Matches ----------
    all_matches = []

    for year in YEARS:
        url = f"https://huggingface.co/datasets/Aneeshers/tennis-sackmann-archive/resolve/main/{tour}/{tour}_matches_{year}.csv"

        try:
            df = pd.read_csv(url)
            df["year"] = year
            all_matches.append(df)

            print(f"Downloaded {tour}_matches_{year}.csv")

        except Exception as e:
            print(f"Could not download {tour} {year}: {e}")

    # ---------- Combine matches ----------
    all_matches_df = pd.concat(
        all_matches,
        ignore_index=True
    )

    for column in ["winner_id", "loser_id"]:
        all_matches_df[column] = pd.to_numeric(all_matches_df[column], errors="coerce").astype("Int64")
        all_matches_df[column] = all_matches_df[column].replace(PLAYER_ID_FIXES[tour])

    all_matches_df["tour"] = tour.upper()

    # Standardize surface values, for example "clay" -> "Clay"
    if "surface" in all_matches_df.columns:
        all_matches_df["surface"] = (
            all_matches_df["surface"]
            .astype("string")
            .str.strip()
            .str.title()
        )

    # Missing hand means unknown, so standardize it as "U"
    for column in ["winner_hand", "loser_hand"]:
        if column in all_matches_df.columns:
            all_matches_df[column] = (
                all_matches_df[column]
                .astype("string")
                .str.strip()
                .str.upper()
                .fillna("U")
                .replace("", "U")
            )

    # Create a unique tournament identifier across ATP and WTA
    all_matches_df["tournament_key"] = all_matches_df["tour"] + "_" + all_matches_df["tourney_id"].astype("string")
    all_matches_df["winner_player_key"] = all_matches_df["tour"] + "_" + all_matches_df["winner_id"].astype("string")
    all_matches_df["loser_player_key"] = all_matches_df["tour"] + "_" + all_matches_df["loser_id"].astype("string")

    for column in ["winner_entry", "loser_entry"]:
        all_matches_df[column] = all_matches_df[column].replace(ENTRY_FIXES)

    # Replace zero-minute values in non-walkover matches with missing values
    invalid_zero_minutes_mask = (all_matches_df["minutes"].eq(0) & ~all_matches_df["score"].isin(["W/O", "Walkover"]))
    print("Non-walkover matches with minutes = 0:", invalid_zero_minutes_mask.sum())

    all_matches_df.loc[
        invalid_zero_minutes_mask,
        "minutes"
    ] = pd.NA

    print(f"\nOriginal matches: {len(all_matches_df)}")
    print("Original tournaments:", all_matches_df["tournament_key"].nunique())

    # ---------- Remove team events ----------
    team_event_mask = (all_matches_df["tourney_level"].eq("D") | all_matches_df["tourney_name"].isin(TEAM_TOURNAMENTS))
    all_matches_df = all_matches_df[~team_event_mask].copy()

    # ---------- Remove explicitly excluded tournaments ----------
    all_matches_df = all_matches_df[~all_matches_df["tourney_name"].isin(EXCLUDED_TOURNAMENTS)].copy()

    # ---------- Remove tournaments with RR / BR ----------
    special_tournament_keys = all_matches_df.loc[all_matches_df["round"].isin(SPECIAL_ROUNDS), "tournament_key"].unique()
    all_matches_df = all_matches_df[~all_matches_df["tournament_key"].isin(special_tournament_keys)].copy()

    print(f"Clean matches: {len(all_matches_df)}")
    print("Clean tournaments:", all_matches_df["tournament_key"].nunique())

    # ---------- Save cleaned matches ----------
    all_matches_df.to_csv(CLEAN_MATCH_FILES[tour], index=False)
    print(f"Saved {CLEAN_MATCH_FILES[tour]}")

    # ---------- Filter active players ----------
    active_player_ids = pd.concat([
        all_matches_df["winner_id"],
        all_matches_df["loser_id"]
    ]).dropna().unique()

    active_players_clean_df = players_df[players_df["player_id"].isin(active_player_ids)].copy()


    # ---------- Save active players ----------
    active_players_clean_df.to_csv(CLEAN_PLAYER_FILES[tour], index=False)

    print(f"Saved {CLEAN_PLAYER_FILES[tour]}")
    print(f"Active players: {len(active_players_clean_df)}")

## Clean Match Data Validation

The cleaned ATP and WTA datasets are validated before tournament-level aggregation.

The checks cover missing values, duplicate player and match identifiers, invalid categorical values, implausible numeric values, tournament-format filtering, and referential integrity between match records and the player tables.

These checks are diagnostic only and do not modify the cleaned datasets.

In [ ]:
# Validate the cleaned player and match datasets

def missing_values_report(df):
    """
    Returns columns that contain missing values,
    together with their counts and percentages.
    """
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (df.isna().mean() * 100).round(2)
    })

    return (
        report[report["missing_count"] > 0]
        .sort_values(
            ["missing_percentage", "missing_count"],
            ascending=False
        )
    )


def show_suspicious_rows(df, mask, columns, title, max_rows=20):
    """
    Displays suspicious rows and returns them.
    """
    existing_columns = [
        column
        for column in columns
        if column in df.columns
    ]

    suspicious_df = df.loc[mask, existing_columns].copy()

    print(f"\n{title}: {len(suspicious_df)} rows")

    if not suspicious_df.empty:
        display(suspicious_df.head(max_rows))

    return suspicious_df


for tour in TOURS:
    print("\n" + "=" * 70)
    print(f"{tour.upper()} DATA VALIDATION")
    print("=" * 70)

    players_df = pd.read_csv(CLEAN_PLAYER_FILES[tour])
    matches_df = pd.read_csv(CLEAN_MATCH_FILES[tour])

    print(f"\nPlayers: {len(players_df):,}")
    print(f"Matches: {len(matches_df):,}")
    print(f"Tournaments: {matches_df['tourney_id'].nunique():,}")

    # =========================================================
    # 1. MISSING VALUES
    # =========================================================

    print("\n--- Missing values: players ---")
    display(missing_values_report(players_df))

    print("\n--- Missing values: matches ---")
    display(missing_values_report(matches_df))

    # =========================================================
    # 2. PLAYER DATA VALIDATION
    # =========================================================

    print("\n--- Player checks ---")

    # Duplicate player identifiers
    duplicate_player_keys = players_df[players_df.duplicated(subset=["player_key"], keep=False)].sort_values("player_key")

    print("Duplicate player_key rows:", len(duplicate_player_keys))

    if not duplicate_player_keys.empty:
        display(duplicate_player_keys)

    duplicate_player_ids = players_df[
        players_df.duplicated(
            subset=["player_id"],
            keep=False
        )
    ].sort_values("player_id")

    print("Duplicate player_id rows:", len(duplicate_player_ids))

    if not duplicate_player_ids.empty:
        display(duplicate_player_ids)

    # Same name associated with multiple IDs
    multiple_ids_per_name = (
        players_df
        .groupby("full_name")["player_id"]
        .nunique()
    )

    multiple_ids_per_name = (
        multiple_ids_per_name[
            multiple_ids_per_name > 1
        ]
    )

    print("Names associated with multiple player IDs:", len(multiple_ids_per_name))

    if not multiple_ids_per_name.empty:
        display(multiple_ids_per_name)

    # Missing or empty names
    empty_name_mask = (
        players_df["full_name"].isna()
        | players_df["full_name"]
        .astype(str)
        .str.strip()
        .eq("")
    )

    show_suspicious_rows(
        players_df,
        empty_name_mask,
        [
            "player_id",
            "player_key",
            "name_first",
            "name_last",
            "full_name"
        ],
        "Players with missing names"
    )

    # Invalid hand values
    if "hand" in players_df.columns:

        invalid_hand_mask = (
            players_df["hand"].notna()
            & ~players_df["hand"].isin(
                ["R", "L", "U"]
            )
        )

        show_suspicious_rows(
            players_df,
            invalid_hand_mask,
            [
                "player_id",
                "full_name",
                "hand"
            ],
            "Players with invalid hand values"
        )

    # Suspicious player heights
    if "height" in players_df.columns:

        height_values = pd.to_numeric(
            players_df["height"],
            errors="coerce"
        )

        suspicious_height_mask = (
            height_values.notna()
            & (
                (height_values < 140)
                | (height_values > 220)
            )
        )

        show_suspicious_rows(
            players_df,
            suspicious_height_mask,
            [
                "player_id",
                "full_name",
                "height",
                "ioc"
            ],
            "Players with suspicious heights"
        )

    # Invalid birth-date format
    if "dob" in players_df.columns:
        dob_as_string = (
            pd.to_numeric(
                players_df["dob"],
                errors="coerce"
            )
            .astype("Int64")
            .astype("string")
        )

        parsed_dob = pd.to_datetime(
            dob_as_string,
            format="%Y%m%d",
            errors="coerce"
        )

        invalid_dob_mask = (
            players_df["dob"].notna()
            & parsed_dob.isna()
        )

        show_suspicious_rows(
            players_df,
            invalid_dob_mask,
            [
                "player_id",
                "full_name",
                "dob"
            ],
            "Players with invalid birth dates"
        )

    # =========================================================
    # 3. DUPLICATE MATCHES
    # =========================================================

    print("\n--- Duplicate match checks ---")

    match_identifier_columns = [
        column
        for column in [
            "tourney_id",
            "tourney_date",
            "match_num",
            "winner_id",
            "loser_id"
        ]
        if column in matches_df.columns
    ]

    duplicate_matches = matches_df[
        matches_df.duplicated(
            subset=match_identifier_columns,
            keep=False
        )
    ].sort_values(match_identifier_columns)

    print("Potential duplicate match rows:", len(duplicate_matches))

    if not duplicate_matches.empty:
        display(duplicate_matches[match_identifier_columns].head(20))

    # =========================================================
    # 4. MATCH DATA VALIDATION
    # =========================================================

    print("\n--- Match checks ---")

    # Winner and loser should not be the same player
    same_player_mask = (
        matches_df["winner_id"].notna()
        & matches_df["loser_id"].notna()
        & matches_df["winner_id"].eq(matches_df["loser_id"])
    )

    show_suspicious_rows(
        matches_df,
        same_player_mask,
        [
            "tourney_id",
            "tourney_name",
            "tourney_date",
            "winner_id",
            "winner_name",
            "loser_id",
            "loser_name"
        ],
        "Matches where winner and loser have the same ID"
    )

    # Only expected surfaces should appear
    invalid_surface_mask = (
        matches_df["surface"].notna()
        & ~matches_df["surface"].isin(
            EXPECTED_SURFACES
        )
    )

    show_suspicious_rows(
        matches_df,
        invalid_surface_mask,
        [
            "tourney_id",
            "tourney_name",
            "tourney_date",
            "surface"
        ],
        "Unexpected surface values"
    )

    # Only standard knockout rounds should remain
    invalid_round_mask = (
        matches_df["round"].notna()
        & ~matches_df["round"].isin(
            EXPECTED_ROUNDS
        )
    )

    show_suspicious_rows(
        matches_df,
        invalid_round_mask,
        [
            "tourney_id",
            "tourney_name",
            "tourney_date",
            "round"
        ],
        "Unexpected round values"
    )

    print("\nRound counts:")
    display(
        matches_df["round"]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
    )

    # Team events should have been removed
    remaining_team_events_mask = (
        matches_df["tourney_level"].eq("D")
        | matches_df["tourney_name"].isin(
            TEAM_TOURNAMENTS
        )
    )

    show_suspicious_rows(
        matches_df,
        remaining_team_events_mask,
        [
            "tourney_id",
            "tourney_name",
            "tourney_level",
            "tourney_date"
        ],
        "Team-event rows remaining after filtering"
    )

    # RR and BR rounds should have been removed
    remaining_special_rounds_mask = (
        matches_df["round"].isin(
            SPECIAL_ROUNDS
        )
    )

    show_suspicious_rows(
        matches_df,
        remaining_special_rounds_mask,
        [
            "tourney_id",
            "tourney_name",
            "round",
            "tourney_date"
        ],
        "RR or BR rows remaining after filtering"
    )

    # Rank values should be positive when present
    for rank_column in ["winner_rank", "loser_rank"]:

        if rank_column not in matches_df.columns:
            continue

        rank_values = pd.to_numeric(matches_df[rank_column], errors="coerce")

        invalid_rank_mask = (rank_values.notna() & (rank_values <= 0))

        show_suspicious_rows(matches_df, invalid_rank_mask,
            [
                "tourney_id",
                "tourney_name",
                "tourney_date",
                "winner_name",
                "loser_name",
                rank_column
            ],
            f"Non-positive values in {rank_column}"
        )

    # Ranking points should not be negative
    for points_column in ["winner_rank_points", "loser_rank_points"]:
        if points_column not in matches_df.columns:
            continue

        points_values = pd.to_numeric(matches_df[points_column], errors="coerce")
        negative_points_mask = (points_values.notna() & (points_values < 0))

        show_suspicious_rows(matches_df, negative_points_mask,
            [
                "tourney_id",
                "tourney_name",
                "tourney_date",
                "winner_name",
                "loser_name",
                points_column
            ],
            f"Negative values in {points_column}"
        )

    # Age should be within a reasonable range
    for age_column in ["winner_age", "loser_age"]:
        if age_column not in matches_df.columns:
            continue

        age_values = pd.to_numeric(matches_df[age_column], errors="coerce")
        suspicious_age_mask = (age_values.notna() & ((age_values < 14) | (age_values > 50)))

        show_suspicious_rows(matches_df, suspicious_age_mask,
            [
                "tourney_id",
                "tourney_name",
                "tourney_date",
                "winner_name",
                "loser_name",
                age_column
            ],
            f"Suspicious values in {age_column}"
        )

    # Height should be within a reasonable range
    for height_column in ["winner_ht", "loser_ht"]:
        if height_column not in matches_df.columns:
            continue

        height_values = pd.to_numeric(matches_df[height_column], errors="coerce")

        suspicious_height_mask = (
            height_values.notna()
            & (
                (height_values < 140)
                | (height_values > 220)
            )
        )

        show_suspicious_rows(
            matches_df,
            suspicious_height_mask,
            [
                "tourney_id",
                "tourney_name",
                "tourney_date",
                "winner_name",
                "loser_name",
                height_column
            ],
            f"Suspicious values in {height_column}"
        )

    # Non-walkover matches with recorded duration should have positive duration
    if "minutes" in matches_df.columns:
        minutes_values = pd.to_numeric(matches_df["minutes"], errors="coerce")

        non_walkover_mask = (
            matches_df["score"].notna()
            & ~matches_df["score"].str.contains(
                r"\bW/?O\b",
                case=False,
                regex=True,
                na=False
            )
        )

        non_positive_minutes_mask = (
            non_walkover_mask
            & minutes_values.notna()
            & (minutes_values <= 0)
        )

        show_suspicious_rows(
            matches_df,
            non_positive_minutes_mask,
            [
                "tourney_id",
                "tourney_name",
                "tourney_date",
                "winner_name",
                "loser_name",
                "score",
                "minutes"
            ],
            "Completed non-walkover matches with non-positive duration"
        )

    # Count statistics cannot be negative
    non_negative_columns = [
        "w_ace",
        "l_ace",
        "w_df",
        "l_df",
        "w_svpt",
        "l_svpt"
    ]

    for column in non_negative_columns:
        if column not in matches_df.columns:
            continue

        values = pd.to_numeric(matches_df[column], errors="coerce")
        negative_mask = (values.notna() & (values < 0))

        show_suspicious_rows(
            matches_df,
            negative_mask,
            [
                "tourney_id",
                "tourney_name",
                "winner_name",
                "loser_name",
                column
            ],
            f"Negative values in {column}"
        )

    # best_of should be 3 or 5 when present
    if "best_of" in matches_df.columns:
        invalid_best_of_mask = (
            matches_df["best_of"].notna()
            & ~matches_df["best_of"].isin(
                [3, 5]
            )
        )

        show_suspicious_rows(
            matches_df,
            invalid_best_of_mask,
            [
                "tourney_id",
                "tourney_name",
                "tourney_date",
                "best_of"
            ],
            "Unexpected best_of values"
        )

    # =========================================================
    # 5. REFERENTIAL INTEGRITY
    # =========================================================

    print("\n--- Players appearing in matches but missing from player table ---")

    match_player_keys = set(
        pd.concat([
            matches_df["winner_player_key"],
            matches_df["loser_player_key"]
        ])
        .dropna()
        .unique()
    )

    known_player_keys = set(players_df["player_key"].dropna())
    missing_player_keys = sorted(match_player_keys - known_player_keys)
    print("Player keys in matches but not in players table:", len(missing_player_keys))

    if missing_player_keys:
        display(pd.DataFrame({"missing_player_key": missing_player_keys}).head(30))

## Player-Tournament Construction Validation

The source data contains one row per match, while the analysis requires one row per player-tournament.

Before aggregating the match data, we validate the assumptions required for this transformation. Tournament-level metadata must be constant within a tournament, each tournament must contain exactly one final, player attributes must be consistent within a player-tournament pair, and the match history must follow the expected knockout structure.

In [ ]:
# ============================================================
# Configuration
# ============================================================

TOURNAMENT_COLUMNS = [
    "tourney_name",
    "surface",
    "draw_size",
    "tourney_level",
    "tourney_date",
]

PLAYER_COLUMNS_TO_CHECK = [
    "player_name",
    "player_ioc",
    "player_hand",
    "player_height",
    "player_age",
    "player_rank",
    "player_rank_points",
    "player_seed",
    "player_entry",
]

ROUND_ORDER = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7,
}


# ============================================================
# General preparation
# ============================================================

def prepare_matches_for_validation(matches_df):
    """
    Prepare the match-level dataset for validation without changing
    the original DataFrame.
    """
    df = matches_df.copy()

    required_columns = [
        "tour",
        "tourney_id",
        "tournament_key",
        "tourney_name",
        "tourney_date",
        "round",
        "winner_id",
        "loser_id",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    # Convert tournament date to datetime
    df["tourney_date"] = pd.to_datetime(
        df["tourney_date"],
        format="%Y%m%d",
        errors="coerce",
    )

    # Extract the tournament year
    df["tourney_year"] = df["tourney_date"].dt.year.astype("Int64")

    # Normalize round labels
    df["round"] = df["round"].astype("string").str.strip().str.upper()

    # Add a numeric round order for sorting
    df["round_order"] = df["round"].map(ROUND_ORDER).astype("Int64")

    return df


def check_tournament_column_consistency(matches_df, tournament_key, columns_to_check,):
    """
    Check whether tournament-level columns have one unique value
    within each tournament.
    """
    print("=" * 70)
    print("TOURNAMENT-LEVEL COLUMN CONSISTENCY")
    print("=" * 70)

    available_columns = [
        column for column in columns_to_check
        if column in matches_df.columns
    ]

    missing_columns = [
        column for column in columns_to_check
        if column not in matches_df.columns
    ]

    if missing_columns:
        print(f"Columns not found and skipped: {missing_columns}")

    all_inconsistencies = []

    for column in available_columns:
        unique_counts = (
            matches_df
            .groupby(tournament_key, dropna=False)[column]
            .nunique(dropna=True)
            .reset_index(name="number_of_unique_values")
        )

        inconsistent = unique_counts[unique_counts["number_of_unique_values"] > 1].copy()
        inconsistent["checked_column"] = column
        all_inconsistencies.append(inconsistent)
        print(f"{column}: {len(inconsistent)} tournaments with multiple values")

    if all_inconsistencies:
        inconsistencies_df = pd.concat(all_inconsistencies, ignore_index=True)
    else:
        inconsistencies_df = pd.DataFrame()

    if not inconsistencies_df.empty:
        print("\nExamples of inconsistent tournament columns:")
        display(inconsistencies_df.head(30))

    return inconsistencies_df


def check_tournament_finals(matches_df, tournament_key):
  """
  Check the number of final matches in each tournament.
  """
  print("=" * 70)
  print("TOURNAMENT FINAL CHECK")
  print("=" * 70)

  final_matches = matches_df[matches_df["round"].eq("F")].copy()

  final_counts = (
      final_matches
      .groupby(tournament_key, dropna=False)
      .size()
      .reset_index(name="number_of_finals")
  )

  all_tournaments = matches_df[tournament_key].drop_duplicates()

  final_counts = all_tournaments.merge(final_counts, on=tournament_key, how="left")

  final_counts["number_of_finals"] = (
      final_counts["number_of_finals"]
      .fillna(0)
      .astype(int)
  )

  invalid_final_counts = final_counts[final_counts["number_of_finals"] != 1].copy()

  print(f"Total tournaments: {len(final_counts)}")
  print(f"Tournaments without exactly one final: {len(invalid_final_counts)}")

  if len(invalid_final_counts) > 0:
      display(invalid_final_counts.head(30))

  return invalid_final_counts


def build_player_match_table(matches_df):
    """
    Convert the match-level winner/loser representation into
    a unified player-match representation.
    """
    shared_columns = [
        column for column in [
            "tour",
            "tournament_key",
            "tourney_id",
            "tourney_year",
            "tourney_name",
            "tourney_date",
            "tourney_level",
            "surface",
            "draw_size",
            "best_of",
            "round",
            "round_order",
            "score",
            "minutes",
        ]
        if column in matches_df.columns
    ]

    winner_mapping = {
        "winner_id": "player_id",
        "winner_name": "player_name",
        "winner_ioc": "player_ioc",
        "winner_hand": "player_hand",
        "winner_ht": "player_height",
        "winner_age": "player_age",
        "winner_rank": "player_rank",
        "winner_rank_points": "player_rank_points",
        "winner_seed": "player_seed",
        "winner_entry": "player_entry",
    }

    loser_mapping = {
        "loser_id": "player_id",
        "loser_name": "player_name",
        "loser_ioc": "player_ioc",
        "loser_hand": "player_hand",
        "loser_ht": "player_height",
        "loser_age": "player_age",
        "loser_rank": "player_rank",
        "loser_rank_points": "player_rank_points",
        "loser_seed": "player_seed",
        "loser_entry": "player_entry",
    }

    available_winner_columns = {
        source: target
        for source, target in winner_mapping.items()
        if source in matches_df.columns
    }

    available_loser_columns = {
        source: target
        for source, target in loser_mapping.items()
        if source in matches_df.columns
    }

    winners = (
        matches_df[
            shared_columns + list(available_winner_columns.keys())
        ]
        .rename(columns=available_winner_columns)
        .copy()
    )

    winners["match_result"] = "win"

    losers = matches_df[shared_columns + list(available_loser_columns.keys())].rename(columns=available_loser_columns).copy()
    losers["match_result"] = "loss"

    player_matches = pd.concat([winners, losers], ignore_index=True,)

    # Create a globally unique ATP/WTA player identifier
    player_matches["player_key"] = (
        player_matches["tour"].astype("string")
        + "_"
        + player_matches["player_id"].astype("Int64").astype("string")
    )

    return player_matches


def check_player_attribute_consistency(player_matches, tournament_key, columns_to_check):
    """
    Check whether player attributes remain constant for the same
    player within the same tournament.
    """
    print("=" * 70)
    print("PLAYER ATTRIBUTE CONSISTENCY WITHIN TOURNAMENT")
    print("=" * 70)

    player_tournament_key = tournament_key + ["player_key"]

    available_columns = [
        column for column in columns_to_check
        if column in player_matches.columns
    ]

    missing_columns = [
        column for column in columns_to_check
        if column not in player_matches.columns
    ]

    if missing_columns:
        print(f"Columns not found and skipped: {missing_columns}")

    all_inconsistencies = []

    for column in available_columns:
        unique_counts = (
            player_matches
            .groupby(player_tournament_key, dropna=False)[column]
            .nunique(dropna=True)
            .reset_index(name="number_of_unique_values")
        )

        inconsistent = unique_counts[unique_counts["number_of_unique_values"] > 1].copy()

        inconsistent["checked_column"] = column
        all_inconsistencies.append(inconsistent)
        print(f"{column}: {len(inconsistent)} player-tournament inconsistencies")

    if all_inconsistencies:
        inconsistencies_df = pd.concat(all_inconsistencies, ignore_index=True,)
    else:
        inconsistencies_df = pd.DataFrame()

    if not inconsistencies_df.empty:
        print("\nExamples:")
        display(inconsistencies_df.head(30))

    return inconsistencies_df


def check_multiple_losses(player_matches, tournament_key):
    """
    Check whether a player has more than one recorded loss
    within the same tournament.
    """
    print("=" * 70)
    print("MULTIPLE LOSSES CHECK")
    print("=" * 70)

    player_tournament_key = tournament_key + ["player_key"]

    loss_counts = (
        player_matches[player_matches["match_result"].eq("loss")]
        .groupby(player_tournament_key, dropna=False)
        .size()
        .reset_index(name="number_of_losses")
    )

    multiple_losses = loss_counts[loss_counts["number_of_losses"] > 1].copy()
    print(f"Players with more than one loss in the same tournament: {len(multiple_losses)}")

    if len(multiple_losses) > 0:
        display(multiple_losses.head(30))

    return multiple_losses

def check_duplicate_player_rounds(player_matches, tournament_key):
    """
    Check whether a player appears more than once in the same
    tournament round.
    """
    print("=" * 70)
    print("DUPLICATE PLAYER-ROUND CHECK")
    print("=" * 70)

    grouping_columns = tournament_key + ["player_key", "round"]

    counts = (
        player_matches
        .groupby(grouping_columns, dropna=False)
        .size()
        .reset_index(name="number_of_appearances")
    )

    duplicates = counts[counts["number_of_appearances"] > 1].copy()

    print(f"Duplicate player-round appearances: {len(duplicates)}")

    if len(duplicates) > 0:
        display(duplicates.head(30))

    return duplicates


def check_tournament_round_structure(matches_df):
    """
    Check that the recorded rounds of each tournament form
    a continuous knockout sequence ending in the final.
    """
    print("=" * 70)
    print("TOURNAMENT ROUND STRUCTURE CHECK")
    print("=" * 70)

    tournament_rounds = (
        matches_df
        .groupby("tournament_key")["round_order"]
        .apply(
            lambda values: sorted(
                values.dropna().astype(int).unique()
            )
        )
    )

    invalid_tournaments = []

    for tournament_key, rounds in tournament_rounds.items():
        if not rounds:
            continue

        expected_rounds = list(range(min(rounds), ROUND_ORDER["F"] + 1))

        if rounds != expected_rounds:
            invalid_tournaments.append({
                "tournament_key": tournament_key,
                "round_orders": rounds,
                "expected_round_orders": expected_rounds,
            })

    invalid_tournaments = pd.DataFrame(invalid_tournaments)

    print("Tournaments with incomplete round structure:", len(invalid_tournaments))

    if not invalid_tournaments.empty:
        display(invalid_tournaments.head(30))

    return invalid_tournaments


In [ ]:
atp_matches_df_clean = pd.read_csv(ATP_CLEAN_MATCHES_FILE)
wta_matches_df_clean = pd.read_csv(WTA_CLEAN_MATCHES_FILE)

matches_df = pd.concat([atp_matches_df_clean, wta_matches_df_clean],ignore_index=True)

# Prepare the dataset
validation_matches = prepare_matches_for_validation(matches_df)

# Use the globally unique tournament identifier
TOURNAMENT_KEY = ["tournament_key"]

# 1. Check tournament-level column consistency
tournament_inconsistencies = (
    check_tournament_column_consistency(
        matches_df=validation_matches,
        tournament_key=TOURNAMENT_KEY,
        columns_to_check=TOURNAMENT_COLUMNS,
    )
)

# 2. Check tournament finals
invalid_finals = check_tournament_finals(
    matches_df=validation_matches,
    tournament_key=TOURNAMENT_KEY,
)

# 3. Create the temporary player-match table
player_matches_df = build_player_match_table(
    validation_matches
)

print("=" * 70)
print("PLAYER-MATCH TABLE")
print("=" * 70)
print(f"Rows: {len(player_matches_df)}")
print(f"Unique players: {player_matches_df['player_key'].nunique()}")
display(player_matches_df.head())

# 4. Check whether player attributes change within a tournament
player_attribute_inconsistencies = (
    check_player_attribute_consistency(
        player_matches=player_matches_df,
        tournament_key=TOURNAMENT_KEY,
        columns_to_check=PLAYER_COLUMNS_TO_CHECK,
    )
)

# 5. Check for multiple losses
multiple_losses = check_multiple_losses(
    player_matches=player_matches_df,
    tournament_key=TOURNAMENT_KEY,
)

# 6. Check for duplicate player appearances in the same round
duplicate_player_rounds = check_duplicate_player_rounds(
    player_matches=player_matches_df,
    tournament_key=TOURNAMENT_KEY,
)

# 7. Check tournament round structure
round_structure_issues = (
    check_tournament_round_structure(
        validation_matches
    )
)


## Player-Tournament Dataset Construction

The original dataset contains one row per match, with separate columns describing the winner and the loser. We transformed it into a player-tournament dataset, where each row represents one player participating in one tournament. Each match was converted into two player-match records, one for the winner and one for the loser, which were then aggregated using the globally unique `player_key` and `tournament_key`.

Tournament-level attributes, such as tournament name, date, surface, level, and draw size, were retained after verifying that they were constant within each tournament. Player attributes, including age, ranking, ranking points, height, nationality, and playing hand, were aggregated using the first available non-missing value, since these attributes were found to be consistent within each player-tournament pair. For `player_seed` and `player_entry`, the most frequent non-missing value was retained to handle a small number of inconsistent source records.

For each player-tournament pair, we calculated the number of matches played, wins, losses, the first and last rounds played, and the final tournament outcome. Players who won the final were labeled as `Winner`, while all other players were assigned the round in which they were eliminated.

Finally, we created a normalized `tournament_finish_score` ranging from 0 to 1. The score is based on the complete round structure of the tournament, where a first-round elimination receives 0, the tournament winner receives 1, and intermediate finish positions are evenly spaced. The score reflects the player's final tournament achievement and is independent of whether the player received a first-round bye.


During validation, one player-tournament record was found to have an unknown final outcome because the original match dataset was missing a match from that tournament. Since the player's final finish position could not be determined from the available data, this single record was excluded from the final dataset.

In [ ]:
# ============================================================
# Configuration
# ============================================================

MATCH_FILES = [ATP_CLEAN_MATCHES_FILE, WTA_CLEAN_MATCHES_FILE]

# ============================================================
# Helper functions
# ============================================================

def first_non_null(series):
    """
    Return the first non-missing value in a Series.
    Return NaN if all values are missing.
    """
    values = series.dropna()
    if values.empty:
        return np.nan
    return values.iloc[0]

def most_common_non_null(series):
    """
    Return the most frequent non-missing value.
    Return NaN if all values are missing.
    """
    values = series.dropna()
    if values.empty:
        return np.nan
    return values.mode().iloc[0]

def get_round_name_by_order(group):
    """
    Return the round name corresponding to the lowest round_order
    in a player-tournament group.
    """
    valid_rows = group.dropna(subset=["round_order"]).sort_values("round_order")
    if valid_rows.empty:
        return pd.NA
    return valid_rows.iloc[0]["round"]


def get_last_round_name(group):
    """
    Return the round name corresponding to the highest round_order
    in a player-tournament group.
    """
    valid_rows = group.dropna(subset=["round_order"]).sort_values("round_order")
    if valid_rows.empty:
        return pd.NA
    return valid_rows.iloc[-1]["round"]


def calculate_finish_stage_index(row):
    """
    Return the player's finish-stage index within the tournament.

    The earliest tournament round receives index 0.
    The tournament winner receives the final index.
    """
    round_orders = row["tournament_round_orders"]

    if not isinstance(round_orders, list):
        return np.nan

    if row["won_tournament"]:
        return len(round_orders)

    if pd.isna(row["loss_round_order"]):
        return np.nan

    try:
        return round_orders.index(int(row["loss_round_order"]))
    except ValueError:
        return np.nan


def get_season(month):
    """
    Return the meteorological season for a given month.
    """
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    if month in [9, 10, 11]:
        return "Fall"

    return pd.NA


# ============================================================
# 1. Load the cleaned match files
# ============================================================

missing_files = [
    path for path in MATCH_FILES
    if not os.path.exists(path)
]

if missing_files:
    raise FileNotFoundError(f"Missing cleaned match files: {missing_files}")

match_dataframes = [pd.read_csv(path) for path in MATCH_FILES]
matches_df = pd.concat(match_dataframes, ignore_index=True)
print(f"Loaded {len(matches_df):,} match rows from {len(MATCH_FILES)} files.")


# ============================================================
# 2. Validate required columns
# ============================================================

required_columns = [
    "tour",
    "tournament_key",
    "tourney_id",
    "tourney_name",
    "tourney_date",
    "tourney_level",
    "surface",
    "draw_size",
    "round",
    "winner_id",
    "winner_player_key",
    "winner_name",
    "winner_ioc",
    "winner_hand",
    "winner_ht",
    "winner_age",
    "winner_rank",
    "winner_rank_points",
    "winner_seed",
    "winner_entry",
    "loser_id",
    "loser_player_key",
    "loser_name",
    "loser_ioc",
    "loser_hand",
    "loser_ht",
    "loser_age",
    "loser_rank",
    "loser_rank_points",
    "loser_seed",
    "loser_entry",
]

missing_columns = [
    column for column in required_columns
    if column not in matches_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")


# ============================================================
# 3. Prepare dates and rounds
# ============================================================

matches_df["tourney_date"] = pd.to_datetime(
    matches_df["tourney_date"],
    format="%Y%m%d",
    errors="coerce"
)

matches_df["tourney_year"] = (
    matches_df["tourney_date"]
    .dt.year
    .astype("Int64")
)

matches_df["season"] = (
    matches_df["tourney_date"]
    .dt.month
    .apply(get_season)
)

matches_df["round"] = (
    matches_df["round"]
    .astype("string")
    .str.strip()
    .str.upper()
)

matches_df["round_order"] = (
    matches_df["round"]
    .map(ROUND_ORDER)
    .astype("Int64")
)

unknown_rounds = (
    matches_df.loc[
        matches_df["round"].notna()
        & matches_df["round_order"].isna(),
        "round"
    ]
    .drop_duplicates()
    .tolist()
)

if unknown_rounds:
    raise ValueError(f"Unknown round values: {unknown_rounds}")


# ============================================================
# 4. Convert every match into two player-match rows
# ============================================================

shared_columns = [
    "tour",
    "tournament_key",
    "tourney_id",
    "tourney_year",
    "season",
    "tourney_name",
    "tourney_date",
    "tourney_level",
    "surface",
    "draw_size",
    "round",
    "round_order",
]

winner_columns = {
    "winner_id": "player_id",
    "winner_player_key": "player_key",
    "winner_name": "player_name",
    "winner_ioc": "player_ioc",
    "winner_hand": "player_hand",
    "winner_ht": "player_height",
    "winner_age": "player_age",
    "winner_rank": "player_rank",
    "winner_rank_points": "player_rank_points",
    "winner_seed": "player_seed",
    "winner_entry": "player_entry",
}

loser_columns = {
    "loser_id": "player_id",
    "loser_player_key": "player_key",
    "loser_name": "player_name",
    "loser_ioc": "player_ioc",
    "loser_hand": "player_hand",
    "loser_ht": "player_height",
    "loser_age": "player_age",
    "loser_rank": "player_rank",
    "loser_rank_points": "player_rank_points",
    "loser_seed": "player_seed",
    "loser_entry": "player_entry",
}

winner_rows = matches_df[shared_columns + list(winner_columns.keys())].rename(columns=winner_columns).copy()
winner_rows["match_result"] = "win"

loser_rows = matches_df[shared_columns + list(loser_columns.keys())].rename(columns=loser_columns).copy()
loser_rows["match_result"] = "loss"

player_matches_df = pd.concat([winner_rows, loser_rows], ignore_index=True)
print(f"Created {len(player_matches_df):,} player-match rows.")

# ============================================================
# 5. Build the tournament round structure
# ============================================================

tournament_round_structure = (
    matches_df[matches_df["round_order"].notna()]
    .groupby("tournament_key")
    .agg(tournament_round_orders=("round_order", lambda values: sorted(values.dropna().astype(int).unique()))
    ).reset_index())

order_to_round = {value: key for key, value in ROUND_ORDER.items()}

tournament_round_structure["tournament_rounds"] = tournament_round_structure["tournament_round_orders"].apply(lambda values: [order_to_round[value] for value in values])

# The recorded rounds end at F.
# Winner is treated as one additional finish state.
tournament_round_structure["number_of_rounds"] = tournament_round_structure["tournament_round_orders"].apply(len)
tournament_round_structure["tournament_first_round"] = tournament_round_structure["tournament_rounds"].str[0]
tournament_round_structure["tournament_last_round"] = tournament_round_structure["tournament_rounds"].str[-1]

# ============================================================
# 6. Build tournament-level metadata
# ============================================================

tournament_metadata = (
    matches_df.groupby("tournament_key", as_index=False)
    .agg(
        tour=("tour", "first"),
        tourney_id=("tourney_id", "first"),
        tourney_year=("tourney_year", "first"),
        season=("season", "first"),
        tourney_name=("tourney_name", "first"),
        tourney_date=("tourney_date", "first"),
        tourney_level=("tourney_level", "first"),
        surface=("surface", "first"),
        draw_size=("draw_size", "first"),
    )
)

tournament_metadata = tournament_metadata.merge(
    tournament_round_structure,
    on="tournament_key",
    how="left"
)


# ============================================================
# 7. Aggregate each player within each tournament
# ============================================================

player_tournament_groups = player_matches_df.groupby(
    ["tournament_key", "player_key"],
    sort=False
)

player_tournament_info = (
    player_tournament_groups
    .agg(
        player_id=("player_id", "first"),
        player_name=("player_name", "first"),
        player_ioc=("player_ioc", first_non_null),
        player_hand=("player_hand", first_non_null),
        player_height=("player_height", first_non_null),
        player_age=("player_age", first_non_null),
        player_rank=("player_rank", first_non_null),
        player_rank_points=(
            "player_rank_points",
            first_non_null
        ),
        player_seed=("player_seed", most_common_non_null),
        player_entry=("player_entry", most_common_non_null),
        matches_played=("match_result", "size"),
        matches_won=(
            "match_result",
            lambda values: values.eq("win").sum()
        ),
        matches_lost=(
            "match_result",
            lambda values: values.eq("loss").sum()
        ),
        first_round_order=("round_order", "min"),
        last_round_order=("round_order", "max"),
    )
    .reset_index()
)

round_summary = (
    player_tournament_groups
    .apply(
        lambda group: pd.Series({
            "first_round_played": get_round_name_by_order(group),
            "last_round_played": get_last_round_name(group),}),
        include_groups=False)
    .reset_index()
)

player_tournament_info = (
    player_tournament_info
    .merge(
        round_summary,
        on=["tournament_key", "player_key"],
        how="left"
    )
)


# ============================================================
# 8. Identify the winner and the loss round
# ============================================================

final_winners = (
    player_matches_df[player_matches_df["round"].eq("F")
        & player_matches_df["match_result"].eq("win")][["tournament_key", "player_key"]]
    .drop_duplicates().assign(won_tournament=True)
)

loss_rounds = (
    player_matches_df[player_matches_df["match_result"].eq("loss")][["tournament_key", "player_key", "round", "round_order"]]
    .rename(columns={"round": "loss_round", "round_order": "loss_round_order",})
)

player_tournament_df = (
    player_tournament_info
    .merge(
        final_winners,
        on=["tournament_key", "player_key"],
        how="left"
    )
    .merge(
        loss_rounds,
        on=["tournament_key", "player_key"],
        how="left"
    )
)

player_tournament_df["won_tournament"] = player_tournament_df["won_tournament"] == True
player_tournament_df["tournament_finish_position"] = np.where(player_tournament_df["won_tournament"], "Winner", player_tournament_df["loss_round"])

# ============================================================
# 9. Add tournament metadata
# ============================================================

player_tournament_df = player_tournament_df.merge(
    tournament_metadata,
    on="tournament_key",
    how="left"
)


# ============================================================
# 10. Calculate the normalized finish score
# ============================================================

player_tournament_df["finish_stage_index"] = (
    player_tournament_df.apply(
        calculate_finish_stage_index,
        axis=1
    )
)

player_tournament_df["tournament_finish_score"] = player_tournament_df["finish_stage_index"] / player_tournament_df["number_of_rounds"]

# ============================================================
# Remove player-tournament records with an unknown final outcome
# ============================================================

# These records correspond to incomplete match histories in the
# original dataset (e.g., missing semifinal or final matches).
# Since the player's final tournament outcome cannot be determined,
# these records are excluded from the final dataset.

unknown_outcome_mask = player_tournament_df["tournament_finish_position"].isna()
print("Removed player-tournament records with unknown outcome:", unknown_outcome_mask.sum())

player_tournament_df = player_tournament_df.loc[~unknown_outcome_mask].reset_index(drop=True)

# ============================================================
# 11. Select and order the final columns
# ============================================================

final_columns = [
    # Identifiers
    "player_key",
    "player_id",
    "tournament_key",

    # Tournament information
    "tour",
    "tourney_id",
    "tourney_year",
    "season",
    "tourney_name",
    "tourney_date",
    "tourney_level",
    "surface",
    "draw_size",
    "tournament_rounds",
    "number_of_rounds",

    # Player information
    "player_name",
    "player_ioc",
    "player_hand",
    "player_height",
    "player_age",
    "player_rank",
    "player_rank_points",
    "player_seed",
    "player_entry",

    # Tournament participation
    "first_round_played",
    "last_round_played",
    "matches_played",
    "matches_won",
    "matches_lost",

    # Tournament outcome
    "won_tournament",
    "tournament_finish_position",
    "finish_stage_index",
    "tournament_finish_score",
]

player_tournament_df = (
    player_tournament_df[final_columns]
    .sort_values(
        [
            "tourney_date",
            "tour",
            "tourney_name",
            "player_name",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 12. Validate the final dataset
# ============================================================

duplicate_count = (
    player_tournament_df
    .duplicated(
        subset=[
            "player_key",
            "tournament_key",
        ]
    )
    .sum()
)

missing_finish_positions = (
    player_tournament_df[
        "tournament_finish_position"
    ]
    .isna()
    .sum()
)

missing_finish_scores = (
    player_tournament_df[
        "tournament_finish_score"
    ]
    .isna()
    .sum()
)

scores_outside_range = (
    player_tournament_df[
        "tournament_finish_score"
    ]
    .notna()
    & ~player_tournament_df[
        "tournament_finish_score"
    ].between(0, 1)
).sum()

winners_with_wrong_score = (
    player_tournament_df["won_tournament"]
    & player_tournament_df[
        "tournament_finish_score"
    ].ne(1)
).sum()

multiple_loss_count = (player_tournament_df["matches_lost"] > 1).sum()
number_of_winners = player_tournament_df["won_tournament"].sum()
number_of_tournaments = player_tournament_df["tournament_key"].nunique()

print("\n" + "=" * 70)
print("PLAYER-TOURNAMENT DATASET VALIDATION")
print("=" * 70)

print(f"Rows: {len(player_tournament_df):,}")
print(f"Tournaments: {number_of_tournaments:,}")
print("Unique players:", player_tournament_df["player_key"].nunique())
print("Duplicate player-tournament rows:", duplicate_count)
print("Missing finish positions:", missing_finish_positions)
print("Missing finish scores:", missing_finish_scores)
print("Finish scores outside [0, 1]:", scores_outside_range)
print("Winners with score different from 1:", winners_with_wrong_score)
print("Players with more than one loss:", multiple_loss_count)
print("Number of tournament winners:", number_of_winners)

assert duplicate_count == 0
assert missing_finish_positions == 0
assert missing_finish_scores == 0
assert scores_outside_range == 0
assert winners_with_wrong_score == 0
assert multiple_loss_count == 0
assert number_of_winners == number_of_tournaments

print("\nValidation passed.")
display(player_tournament_df.head(20))

# ============================================================
# 13. Save the final dataset
# ============================================================
player_tournament_df.to_csv(PLAYER_TOURNAMENT_FILE, index=False)
print(f"\nSaved player-tournament dataset to: {PLAYER_TOURNAMENT_FILE}")